# Modul 19: Sequenzmodelle, Autoencoder und Generierung | Übungen

## Überblick

Sie erstellen zeitlich korrekte Sequenzfenster und vergleichen naive Baselines mit kleinen Conv1D-, SimpleRNN- und GRU-Modellen. Danach trainieren Sie einen dichten Denoising-Autoencoder, analysieren Rekonstruktionsfehler, erkennen ungewöhnliche Ziffern und untersuchen Interpolation im latenten Raum.

**Zugehörige Vorlesungen**

- **Sequenzmodelle**
- **Autoencoder und Generierung**

## Lernziele

Nach der Bearbeitung können Sie:

- Sequenzfenster mit passenden Batchformen, zeitlichen Splits und naiven Baselines vorbereiten.
- kleine Conv1D-, SimpleRNN- und GRU-Modelle CPU-freundlich trainieren und zeitlich auswerten.
- Autoencoder für Rekonstruktion und Denoising einsetzen sowie latente Darstellungen und Anomaliegrenzen kritisch prüfen.

## Geprüfte Fähigkeiten

- Zeitfenster, Persistenzbaseline, zeitliche Skalierung und Fehleranalyse
- Keras Conv1D, SimpleRNN, GRU, Masking und EarlyStopping
- MLP-Autoencoder, Rekonstruktionsfehler, Denoising, Anomalieschwelle und latente Interpolation

## Hinweise zur Bearbeitung

Dieses Notebook dient als praktische Übung und Lernstandskontrolle. Führen Sie zuerst die Einrichtungszelle aus und bearbeiten Sie danach die Aufgaben in der angegebenen Reihenfolge. Die vorgesehenen Arbeitsbereiche sind deutlich markiert.

- **Erwarteter Schwierigkeitsgrad:** anspruchsvoll
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle erzeugt eine lokale univariate Zeitreihe mit Trend, Saison und Rauschen und lädt außerdem die kleinen Digits-Bilder. Die Deep-Learning-Modelle sind bewusst klein und auf wenige Epochen begrenzt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPRegressor

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

# Lokale Zeitreihe mit langsamem Trend, zwei Perioden und Rauschen.
anzahl_zeitpunkte = 1800
zeit_index = np.arange(anzahl_zeitpunkte)
zeitreihe = (
    0.0015 * zeit_index
    + 1.2 * np.sin(2 * np.pi * zeit_index / 50)
    + 0.35 * np.sin(2 * np.pi * zeit_index / 13)
    + rng.normal(0.0, 0.18, size=anzahl_zeitpunkte)
).astype("float32")

train_ende = int(0.60 * anzahl_zeitpunkte)
val_ende = int(0.80 * anzahl_zeitpunkte)

# Die Skalierungsparameter stammen ausschließlich aus der frühen Trainingsperiode.
zeit_mittel = float(zeitreihe[:train_ende].mean())
zeit_std = float(zeitreihe[:train_ende].std())
zeitreihe_skaliert = (zeitreihe - zeit_mittel) / zeit_std

ziffern = load_digits()
digit_bilder = (ziffern.data.astype("float32") / 16.0)
digit_labels = ziffern.target.astype("int64")

print("Zeitreihe:", zeitreihe.shape)
print("Digits-Matrix:", digit_bilder.shape)

### Aufgabe 1: Zeitliche Fenster und Persistenzbaseline erstellen

Implementieren Sie `make_windows`, die für jeden Zielzeitpunkt die vorherigen 30 Werte als Eingabefenster und den nächsten Wert als Ziel erzeugt. Die Funktion erhält einen Zielbereich `[start, end)` und darf nur vergangene Werte verwenden.

Erstellen Sie Training, Validierung und Test anhand der vorgegebenen Zeitgrenzen. Ergänzen Sie die letzte Achse, sodass Keras die Form `(Batch, Schritte, Merkmale)` erhält. Berechnen Sie auf Validierung und Test eine Persistenzbaseline, die einfach den letzten Fensterwert vorhersagt. Berichten Sie MAE in skalierten und ursprünglichen Einheiten.

In [ ]:
def make_windows(series, start, end, window_size):
    pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum darf die Skalierung nicht auf der vollständigen Zeitreihe angepasst werden?

### Aufgabe 2: Ein kleines Conv1D-Prognosemodell trainieren

Erstellen Sie ein Keras-Modell aus einer Conv1D-Schicht mit höchstens 16 Filtern und Kernelgröße 3, GlobalAveragePooling1D, einer kleinen Dense-Schicht und einer linearen Ausgabe. Trainieren Sie höchstens 20 Epochen mit Adam, MSE und MAE sowie EarlyStopping.

Vergleichen Sie Validierungs- und Test-MAE mit der Persistenzbaseline. Zeichnen Sie für die ersten 150 Testzeitpunkte Wahrheit, Baseline und Conv1D-Prognose in ursprünglichen Einheiten.

In [ ]:
# Eingabeform eines Beispiels: (fensterlaenge, 1)

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Welche Muster kann Conv1D gut erfassen und welche Grenze besitzt GlobalAveragePooling?

### Aufgabe 3: SimpleRNN und GRU unter gleichen Bedingungen vergleichen

Erstellen Sie zwei Modelle mit identischer Eingabe und ungefähr ähnlicher kleiner Größe:

- SimpleRNN mit 16 Einheiten,
- GRU mit 16 Einheiten.

Beide erhalten eine lineare Ausgabe und werden mit denselben Splits, Batchgrößen, maximalen Epochen und EarlyStopping-Regeln trainiert. Vergleichen Sie Parameterzahl, ausgeführte Epochen, Validierungs-MAE und Test-MAE. Wählen Sie das Modell ausschließlich anhand der Validierung und berichten Sie danach dessen Testfehler.

In [ ]:
def baue_rekurrentes_modell(art):
    pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum besitzt eine GRU meist mehr Parameter als eine SimpleRNN-Schicht mit gleicher Einheitenzahl?

### Aufgabe 4: Maskierung und zeitliche Fehleranalyse anwenden

Erzeugen Sie einen kleinen Klassifikationsdatensatz aus Sequenzen unterschiedlicher Länge zwischen 12 und 30. Klasse 0 soll überwiegend eine niedrige Frequenz, Klasse 1 eine höhere Frequenz enthalten. Füllen Sie alle Sequenzen rechts mit Nullen auf Länge 30 auf.

Trainieren Sie ein Modell aus `Masking(mask_value=0.0)`, GRU und Sigmoid. Teilen Sie die Sequenzen reproduzierbar und stratifiziert. Vergleichen Sie Test-Accuracy für kurze Sequenzen bis Länge 20 und längere Sequenzen. Erklären Sie, warum echte Messwerte nicht zufällig genau dem Maskierungswert entsprechen sollten.

In [ ]:
# Erzeugen Sie 400 variable Sequenzen und speichern Sie zusätzlich ihre ursprünglichen Längen.

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Welche Gefahr entsteht, wenn der Maskierungswert auch ein gültiger Messwert ist?

### Aufgabe 5: Einen dichten Denoising-Autoencoder trainieren

Teilen Sie die Digits-Daten reproduzierbar in Training, Validierung und Test. Entfernen Sie für das Autoencoder-Training alle Ziffern 9, damit sie später als unbekannte Muster dienen können.

Erzeugen Sie verrauschte Eingaben durch additives Gaußrauschen und Begrenzung auf 0 bis 1. Trainieren Sie einen `MLPRegressor` mit symmetrischer Architektur `(32, 12, 32)`, der aus verrauschten Bildern die sauberen Pixel rekonstruiert. Verwenden Sie Early Stopping und eine begrenzte Iterationszahl. Vergleichen Sie den Rekonstruktions-MSE auf sauberen und verrauschten Testeingaben und visualisieren Sie Original, verrauschte Eingabe und Rekonstruktion für fünf normale Ziffern.

In [ ]:
# Ziffer 9 wird nur aus dem Autoencoder-Training entfernt, nicht aus dem späteren Test.

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum ist ein Autoencoder nicht automatisch ein leistungsfähiges generatives Modell?

### Aufgabe 6: Integrationsaufgabe: Rekonstruktionsanomalien und latente Interpolation

Berechnen Sie für normale Validierungsbilder den mittleren quadratischen Rekonstruktionsfehler pro Bild und setzen Sie die Anomalieschwelle auf das 95. Perzentil. Wenden Sie diese Schwelle auf alle Testbilder an und behandeln Sie Ziffer 9 als positive Anomalieklasse.

Berichten Sie Precision, Recall und F1 der Anomalieerkennung und visualisieren Sie Beispiele mit besonders hohem Fehler. Berechnen Sie anschließend die 12-dimensionale Engpassdarstellung zweier normaler Testbilder manuell aus den gelernten Gewichtsmatrizen, interpolieren Sie fünf Zwischenpunkte und dekodieren Sie sie durch die verbleibenden Schichten. Kennzeichnen Sie klar, dass plausible Interpolation keine Garantie für gültige neue Daten ist.

In [ ]:
def relu_numpy(x):
    return np.maximum(0.0, x)

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Welche Grenzen hat die Anomalieentscheidung über Rekonstruktionsfehler?

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?